# A chat app interface
A chat app with FastHTML and claudette/lisette

- User enters prompt, the chat bubble for the user shows.
- LLM processes. And returns with the response in an AI chat bubble.

In [ ]:
#| default_exp chat

In [ ]:
#| export
from fasthtml.common import *
from lisette import *

In [ ]:
#| hide
from IPython.display import HTML
from fasthtml.jupyter import *

## Creating the app with daisy styling -

In [ ]:
#| export
app = FastHTML(hdrs=(Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'), Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'), Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css')))

In [ ]:
#| export
# For convenience
rt = app.route

In [ ]:
#| hide
#| eval: false
srv = JupyUvi(app)

## LLM system prompt -

In [ ]:
#| export
sysp = "You are a chat model, in a chat with a user. Given the chat history as context, answer the user's current question concisely."

## Previewing components -

#| hide

Convenience function for previewing components

In [ ]:
#| hide
def preview(c):
    return HTMX(c, app=app, host=None, port=None)


In [ ]:
#| hide
@patch
def __html__(self:FT):
    return preview(self).data

## Chat with llm

In [ ]:
#| export
def bubble(msg:str, isAI:bool=False):
    if isAI:
        return Div(Div(msg, cls='chat-bubble chat-bubble-secondary'), cls='chat chat-end')
    else:
        return Div(Div(msg, cls='chat-bubble chat-bubble-primary'), cls='chat chat-start')

In [ ]:
bubble("Can you help?")

<div class="chat chat-start"><div class="chat-bubble chat-bubble-primary">Can you help?</div></div>

In [ ]:
bubble("No", True)

<div class="chat chat-end"><div class="chat-bubble chat-bubble-secondary">No</div></div>

In [ ]:
#| export
def mk_prompt(msg, h):
    return "Past context/conversation: " + "\n".join(h) + "\n\nCurrent question: " + msg

In [ ]:
mk_prompt("Hello", ["Old message"])

'Past context/conversation: Old message\n\nCurrent question: Hello'

## Routes

In [ ]:
#| export
# We need a way to store the history of the conversation.
h = []

In [ ]:
#| export
@rt
def ask_llm(msg:str):
    global h
    r = Chat(model='claude-haiku-4-5', sp=sysp)(mk_prompt(msg, h))
    rsp = r.choices[0].message.content
    h += [str(msg), str(rsp) if rsp else ""]
    return bubble(rsp, True)

In [ ]:
#ask_llm("What's the weather?")

In [ ]:
#| export
def llm_rsp(msg):
    return Div(
        Span(cls='loading loading-ring loading-md'), cls='flex justify-center',
        hx_post=ask_llm, hx_trigger='load', hx_swap='outerHTML',
        hx_vals=f'{{"msg": "{msg}"}}'
        )

In [ ]:
#| export
@rt
def send(msg: str):
    return (bubble(msg), llm_rsp(msg))

Creating the actual chat

In [ ]:
#| export
@rt
def chatpg():
    return Titled("Chat Demo",
        Div(id='chat', style='max-width:600px; margin:auto; border: 1px solid #ccc; border-radius: 0.5rem; height: 400px; overflow-y: auto; padding 1rem'),
        Form(
            Input(id='msg', type='text', placeholder='Type something...', style='flex:1'),
            Button('Send', hx_post=send, hx_target='#chat', hx_swap='beforeend',),
            style='display:flex; gap:8px; max-width:600px; margin:auto; padding:1rem'
        ),
        style='text-align:center'
    )

In [ ]:
#preview(chatpg)

## Adding a note bubble

In [ ]:
#| hide
#| eval: false
srv.stop()

In [ ]:
#| hide
from nbdev import nbdev_export; nbdev_export()